In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, StringType, DoubleType,DateType,BooleanType
from pyspark.sql.functions import round

In [0]:
configs = {
"fs.azure.account.auth.type.tokyoolympicdatavishnu.dfs.core.windows.net": "OAuth",
"fs.azure.account.oauth.provider.type.tokyoolympicdatavishnu.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
"fs.azure.account.oauth2.client.id.tokyoolympicdatavishnu.dfs.core.windows.net": "",
"fs.azure.account.oauth2.client.secret.tokyoolympicdatavishnu.dfs.core.windows.net": "",
"fs.azure.account.oauth2.client.endpoint.tokyoolympicdatavishnu.dfs.core.windows.net": "https://login.microsoftonline.com//oauth2/token"
}

In [0]:
for key, value in configs.items():
    spark.conf.set(key, value)

In [0]:
dbutils.fs.ls("abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/")

[FileInfo(path='abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/', name='RawData/', size=0, modificationTime=1772639365000),
 FileInfo(path='abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/TransformedData/', name='TransformedData/', size=0, modificationTime=1772639376000)]

In [0]:
dbutils.fs.ls("abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/")

[FileInfo(path='abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/Teams.csv', name='Teams.csv', size=2414, modificationTime=1772643022000),
 FileInfo(path='abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/athletes.csv', name='athletes.csv', size=418492, modificationTime=1772642952000),
 FileInfo(path='abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/coaches.csv', name='coaches.csv', size=16889, modificationTime=1772642359000),
 FileInfo(path='abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/genders.csv', name='genders.csv', size=1123, modificationTime=1772642987000),
 FileInfo(path='abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/medals.csv', name='medals.csv', size=2414, modificationTime=1772643004000)]

In [0]:
spark

In [0]:
athletes = spark.read.format("csv").option("header",True).option("InferSchema",True).load("abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/athletes.csv")

teams = spark.read.format("csv").option("header",True).option("InferSchema",True).load("abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/Teams.csv")

coaches = spark.read.format("csv").option("header",True).option("InferSchema",True).load("abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/coaches.csv")

genders = spark.read.format("csv").option("header",True).option("InferSchema",True).load("abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/genders.csv")

medals = spark.read.format("csv").option("header",True).option("InferSchema",True).load("abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/RawData/medals.csv")

In [0]:
athletes.show()

+--------------------+--------------------+-------------------+
|          PersonName|             Country|         Discipline|
+--------------------+--------------------+-------------------+
|     AALERUD Katrine|              Norway|       Cycling Road|
|         ABAD Nestor|               Spain|Artistic Gymnastics|
|   ABAGNALE Giovanni|               Italy|             Rowing|
|      ABALDE Alberto|               Spain|         Basketball|
|       ABALDE Tamara|               Spain|         Basketball|
|           ABALO Luc|              France|           Handball|
|        ABAROA Cesar|               Chile|             Rowing|
|       ABASS Abobakr|               Sudan|           Swimming|
|    ABBASALI Hamideh|Islamic Republic ...|             Karate|
|       ABBASOV Islam|          Azerbaijan|          Wrestling|
|        ABBINGH Lois|         Netherlands|           Handball|
|         ABBOT Emily|           Australia|Rhythmic Gymnastics|
|       ABBOTT Monica|United States of .

In [0]:
athletes.printSchema()

root
 |-- PersonName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)



In [0]:
genders.printSchema()  #Counts are displaying as string, we need to convert to integer to perform further any operations

root
 |-- Discipline: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: string (nullable = true)
 |-- Total: string (nullable = true)



In [0]:
genders=genders.withColumn("Female",col("Female").cast(IntegerType())).withColumn("Male",col("Male").cast(IntegerType())).withColumn("Total",col("Total").cast(IntegerType()))
genders.printSchema()



root
 |-- Discipline: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: integer (nullable = true)
 |-- Total: integer (nullable = true)



In [0]:
medals.show()

+----+--------------------+----+------+------+-----+-------------+
|Rank|         TeamCountry|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   3|               Japan|  27|    14|    17|   58|            5|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   5|                 ROC|  20|    28|    23|   71|            3|
|   6|           Australia|  17|     7|    22|   46|            6|
|   7|         Netherlands|  10|    12|    14|   36|            9|
|   8|              France|  10|    12|    11|   33|           10|
|   9|             Germany|  10|    11|    16|   37|            8|
|  10|               Italy|  10|    10|    20|   40|            7|
|  11|              Canada|   7|     6|    11|   24|           11|
|  12|              Brazil|   7|     6|     8|   21|          

In [0]:
medals.printSchema()

root
 |-- Rank: integer (nullable = true)
 |-- TeamCountry: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Rank by Total: integer (nullable = true)



In [0]:
teams.show()

+----+--------------------+----+------+------+-----+-------------+
|Rank|         TeamCountry|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   3|               Japan|  27|    14|    17|   58|            5|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   5|                 ROC|  20|    28|    23|   71|            3|
|   6|           Australia|  17|     7|    22|   46|            6|
|   7|         Netherlands|  10|    12|    14|   36|            9|
|   8|              France|  10|    12|    11|   33|           10|
|   9|             Germany|  10|    11|    16|   37|            8|
|  10|               Italy|  10|    10|    20|   40|            7|
|  11|              Canada|   7|     6|    11|   24|           11|
|  12|              Brazil|   7|     6|     8|   21|          

In [0]:
#Find Top Countries that have highest number of gold medals
top_gold_medal_countries=medals.orderBy(col("Gold").desc()).select('TeamCountry','Gold').limit(5)
top_gold_medal_countries.show()


+--------------------+----+
|         TeamCountry|Gold|
+--------------------+----+
|United States of ...|  39|
|People's Republic...|  38|
|               Japan|  27|
|       Great Britain|  22|
|                 ROC|  20|
+--------------------+----+



In [0]:
#avg number of entries by gender in each discipline

avg_entries_by_gender=genders.withColumn('Avg_Female',round(genders['Female']/genders['Total'],2)).withColumn('Avg_Male',round(genders['Male']/genders['Total'],2)).select('Discipline','Avg_Female','Avg_Male')
avg_entries_by_gender.show()

+--------------------+----------+--------+
|          Discipline|Avg_Female|Avg_Male|
+--------------------+----------+--------+
|      3x3 Basketball|       0.5|     0.5|
|             Archery|       0.5|     0.5|
| Artistic Gymnastics|       0.5|     0.5|
|   Artistic Swimming|       1.0|     0.0|
|           Athletics|      0.47|    0.53|
|           Badminton|       0.5|     0.5|
|   Baseball/Softball|      0.38|    0.62|
|          Basketball|       0.5|     0.5|
|    Beach Volleyball|       0.5|     0.5|
|              Boxing|      0.35|    0.65|
|        Canoe Slalom|       0.5|     0.5|
|        Canoe Sprint|      0.49|    0.51|
|Cycling BMX Frees...|      0.53|    0.47|
|  Cycling BMX Racing|       0.5|     0.5|
|Cycling Mountain ...|       0.5|     0.5|
|        Cycling Road|      0.35|    0.65|
|       Cycling Track|      0.48|    0.52|
|              Diving|       0.5|     0.5|
|          Equestrian|      0.37|    0.63|
|             Fencing|       0.5|     0.5|
+----------

In [0]:
athletes.write.mode("overwrite").option("header", "true").csv(
"abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/TransformedData/athletes_transformed"
)

In [0]:
athletes.show()

+--------------------+--------------------+-------------------+
|          PersonName|             Country|         Discipline|
+--------------------+--------------------+-------------------+
|     AALERUD Katrine|              Norway|       Cycling Road|
|         ABAD Nestor|               Spain|Artistic Gymnastics|
|   ABAGNALE Giovanni|               Italy|             Rowing|
|      ABALDE Alberto|               Spain|         Basketball|
|       ABALDE Tamara|               Spain|         Basketball|
|           ABALO Luc|              France|           Handball|
|        ABAROA Cesar|               Chile|             Rowing|
|       ABASS Abobakr|               Sudan|           Swimming|
|    ABBASALI Hamideh|Islamic Republic ...|             Karate|
|       ABBASOV Islam|          Azerbaijan|          Wrestling|
|        ABBINGH Lois|         Netherlands|           Handball|
|         ABBOT Emily|           Australia|Rhythmic Gymnastics|
|       ABBOTT Monica|United States of .

In [0]:
medals.repartition(1).write.mode("overwrite").option("header", "true").csv(
"abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/TransformedData/medals_transformed"
)
#repartition will divide the data into 3 partitions

In [0]:
teams.repartition(1).write.mode("overwrite").option("header", "true").csv(
"abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/TransformedData/teams_transformed"
)

In [0]:
coaches.repartition(1).write.mode("overwrite").option("header", "true").csv(
"abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/TransformedData/coaches_transformed"
)

In [0]:
genders.repartition(1).write.mode("overwrite").option("header", "true").csv(
"abfss://tokyo-olympic-data@tokyoolympicdatavishnu.dfs.core.windows.net/TransformedData/genders_transformed"
)